In [ ]:
from qiskit import *
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.primitives import BackendEstimatorV2 
from qiskit_aer import Aer

from functools import partial
import torch
from torch.autograd import Function
import numpy as np
import matplotlib.pyplot as plt
import quimb

import random


In [ ]:
backend = Aer.get_backend('aer_simulator')

def Energy(circuit, lamb=0.0, delta=1.0):
    n = circuit.num_qubits  # number of qubits in the circuit
    coeffs = []
    paulis = []

    # lambda * sum_i Z_i term
    for i in range(n):
        pauli_string = ['I'] * n
        pauli_string[i] = 'Z'
        pauli = Pauli(''.join(pauli_string))
        coeffs.append(lamb)
        paulis.append(pauli)

    # sum_i (X_i X_{i+1} + Y_i Y_{i+1}) + delta * sum_i Z_i Z_{i+1} term
    for i in range(n):
        next_i = (i + 1) % n  # apply periodic boundary conditions

        # X_i X_{i+1} term
        pauli_string = ['I'] * n
        pauli_string[i] = 'X'
        pauli_string[next_i] = 'X'
        pauli = Pauli(''.join(pauli_string))
        coeffs.append(1.0)
        paulis.append(pauli)

        # Y_i Y_{i+1} term
        pauli_string = ['I'] * n
        pauli_string[i] = 'Y'
        pauli_string[next_i] = 'Y'
        pauli = Pauli(''.join(pauli_string))
        coeffs.append(1.0)
        paulis.append(pauli)

        # delta Z_i Z_{i+1} term
        pauli_string = ['I'] * n
        pauli_string[i] = 'Z'
        pauli_string[next_i] = 'Z'
        pauli = Pauli(''.join(pauli_string))
        coeffs.append(delta)
        paulis.append(pauli)

    # Build Hamiltonian
    H = SparsePauliOp(paulis, coeffs=np.array(coeffs))
    
    # Use StatevectorEstimator
    estimator = BackendEstimatorV2(backend=backend)
    shots = 2048
    # Compute expectation value , 
    pub = (circuit, H)
    job = estimator.run([pub], precision = 1 / np.sqrt(shots))
    result = job.result()[0]
    e = result.data.evs
    return e


### Multi-scale entangled randomized ansatz (MERA)

In [ ]:
def HEA(inp, n, d=1, lamb=1.0, energy_flag=False, param_num=False):
    params = inp["params"]
    delta = inp["delta"]
    qc = QuantumCircuit(n)

    idx = 0

    for i in range(n):
        qc.rx(params[3 * i],i)
        qc.rz(params[3 * i + 1],i)
        qc.rx(params[3 * i + 2],i)
    idx += 3 * n

    for _ in range(d):
        for i in range(0, n):
            qc.rzz(params[idx], i, (i + 1) % n)
            idx += 1

        for i in range(0, n):
            qc.rxx(params[idx], i, (i + 1) % n)
            idx += 1

        for i in range(0, n):
            qc.ryy(params[idx], i, (i + 1) % n)
            idx += 1

        for i in range(n):
            qc.rx(params[idx],i)
            qc.rz(params[idx + 1],i)
            idx += 2

    if energy_flag:
        e = Energy(qc, lamb, delta)
        return e
    elif param_num:
        return qc, idx
    else:
        return qc

#### Ansatz circuit

#### compute gradient via parameter shift rule

In [ ]:
def compute_gradients(params_np, delta, n, d, lamb):
    shift = np.pi / 2
    num_params = len(params_np)
    grad_params = np.zeros(num_params)

    for i in range(num_params):
        shifted_params_plus = np.copy(params_np)
        shifted_params_minus = np.copy(params_np)
        shifted_params_plus[i] += shift
        shifted_params_minus[i] -= shift

        # Build quantum circuit
        qc_plus = HEA({"params": shifted_params_plus, "delta": delta}, n, d, lamb)
        qc_minus = HEA({"params": shifted_params_minus, "delta": delta}, n, d, lamb)

        # Compute energy
        energy_plus = Energy(qc_plus, lamb, delta)
        energy_minus = Energy(qc_minus, lamb, delta)

        # Compute gradients
        grad_params[i] = 0.5 * (energy_plus - energy_minus)
    return grad_params

## NN-VQE

Design the NN-VQE. We use a neural network to transform the Hamiltonian parameters to the optimized parameters in the parameterized quantum circuit (PQC) for VQE.

In [ ]:
class QuantumCircuitFunction(torch.autograd.Function):
    grad_params_buffer = None

    @staticmethod
    def forward(ctx, params, delta, n, d, lamb):
        ctx.save_for_backward(params)
        ctx.n = n
        ctx.d = d
        ctx.lamb = lamb

        # Convert delta to float and store it
        if isinstance(delta, torch.Tensor):
            delta_np = delta.detach().cpu().numpy()
            if delta_np.size == 1:
                delta_np = delta_np.item()
            else:
                delta_np = delta_np[0]
        else:
            delta_np = delta

        ctx.delta = delta_np  # store delta as a float

        params_np = params.detach().cpu().numpy()

        qc = HEA({"params": params_np, "delta": delta_np}, n, d, lamb)
        energy = Energy(qc, lamb, delta_np)

        return torch.tensor(energy, dtype=params.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        params, = ctx.saved_tensors
        n = ctx.n
        d = ctx.d
        lamb = ctx.lamb
        delta_np = ctx.delta  # delta is a float

        params_np = params.detach().cpu().numpy()

        grad_params_np = compute_gradients(params_np, delta_np, n, d, lamb)
        grad_params = torch.from_numpy(grad_params_np).to(params.device).float()
        QuantumCircuitFunction.grad_params_buffer = grad_params_np

        return grad_output * grad_params, None, None, None, None


class NN_MERA_Model(torch.nn.Module):
    def __init__(self, n, d, lamb, stddev, NN_shape):
        super(NN_MERA_Model, self).__init__()
        self.n = n
        self.d = d
        self.lamb = lamb
        self.NN_shape = NN_shape

        # Compute the number of parameters needed by MERA
        # Use param_num=True to get idx
        _, idx = HEA({"params": np.random.uniform(0, 2*np.pi, size=1000), "delta": 0}, n, d, lamb, param_num=True)
        self.idx = idx

        # Define neural network layers
        self.hidden_layer = torch.nn.Linear(1, NN_shape)
        torch.nn.init.normal_(self.hidden_layer.weight, mean=0.0, std=stddev)
        self.output_layer = torch.nn.Linear(NN_shape, self.idx)
        torch.nn.init.normal_(self.output_layer.weight, mean=0.0, std=stddev)
        
    def forward(self, delta):
        # x = self.input_layer(delta)
        # x = torch.relu(x)
        x = self.hidden_layer(delta)
        x = torch.relu(x)
        x = self.output_layer(x)
        x = torch.sigmoid(x)
        params = x * 6.3  # Scale parameters as needed

        # Remove the batch dimension to make a 1D tensor.
        params = params.view(-1)
        energy = QuantumCircuitFunction.apply(params, delta, self.n, self.d, self.lamb)
        return energy

## Train

In [ ]:
sum_of_energy = []

def train(n, d, lamb, delta_values, NN_shape, maxiter=1, lr=0.1, stddev=1.0, index=0):
    model = NN_MERA_Model(n, d, lamb, stddev, NN_shape)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    decay_steps = 700
    decay_rate = 0.7
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: decay_rate ** (step // decay_steps)
    )
    
    for i in range(1, maxiter + 1):
        optimizer.zero_grad()
        total_energy = torch.tensor(0.0, dtype=torch.float32)
        
        for delta in delta_values:
            delta_tensor = torch.tensor([[delta]], dtype=torch.float32)  # Shape (1, 1)
            energy = model(delta_tensor)
            total_energy += energy
            
        total_energy.backward()
        grad_param = QuantumCircuitFunction.grad_params_buffer
        optimizer.step()
        scheduler.step()
        sum_of_energy.append(total_energy.item())
        if i % 10 == 0:
            print(f"Epoch {i}, Total Energy: {total_energy.item()}")
    
    return grad_param


In [ ]:
# Define training parameters
n_list = [3,4,5,6,7,8,9]          # Number of qubits
d = [1,2,3,4,5,6,7]          # Circuit depth
lamb = 0.0    # Lambda parameter
NN_shape = 20  # Hidden layer size in the neural network
delta_values = [1.0]  # Example delta values
maxiter = 1
stddev = 1.0
lr=0.005

iter = 500
results_dict = {}  # Dictionary storing repeated scalar results for each n
var_dict = {}
var_mean_list = []

# Start training
for n in n_list:
  result_for_this_n = []
  _, param_num = HEA({"params": np.zeros(1000), "delta": 0.0}, n, d[n-3], param_num=True)
  print("qubit: {}, # of parameter: {}".format(n, param_num))

  index = random.randrange(0,param_num)
  for i in range(iter):
    a = train(n, d[n-3], lamb, delta_values, NN_shape, maxiter, 0.1, stddev, index)
    result_for_this_n.append(a)
  var_list_n=[]
  for j in range(param_num):
      temp_gar = []
      for i in range(iter):
          temp_gar.append(result_for_this_n[i][j])
      var_list_n.append(np.var(temp_gar, ddof=1))
  print(np.mean(var_list_n))

  var_mean_list.append(np.mean(var_list_n))
  var_dict[n] = var_list_n
  results_dict[n] = result_for_this_n


  results_dict[n] = result_for_this_n


In [ ]:
print("NN_var_avg = ", var_mean_list)

In [ ]:
print("NN_var_dict = ", var_dict)

In [ ]:
plt.plot(n_list,var_mean_list)
plt.legend()
plt.show()


In [ ]:

stds = [np.std(var_dict[n]) for n in n_list]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(n_list, var_mean_list, linewidth=2, label='Mean')
plt.fill_between(n_list, np.array(var_mean_list) - np.array(stds), np.array(var_mean_list) + np.array(stds), alpha=0.2)

xtick_labels = [f"{val}({val-2})" for val in n_list]
plt.xticks(n_list, xtick_labels)
plt.ylabel("var[grad.]")
plt.xlabel("# of qubit (ansatz depth)")
plt.title('NN')
plt.legend()
plt.tight_layout()
plt.show()